# 08 · B1 + B2 trên **toàn bộ** ViTextVQA test (10.028 câu), GPU L4 (W7)

**Vấn đề:** B1 và B2 trên ViTextVQA mới chấm trên **tập con 2.000 câu** (seed 42). Kết quả hiện có:

| Run | EM | F1 (phụ) | ANLS |
|---|---|---|---|
| B1 | 28,80 | 53,77 | 47,53 |
| B2 | 49,70 | 71,01 | 71,91 |

SOTA của paper ViTextVQA là **ViTextBLIP-2**: EM 25,48 · F1 53,95 trên **full test** (Table 6, Nguyen et al., *ESWA* 2025, arXiv:2404.10652v5). Hai con số chấm trên hai tập khác nhau nên **chưa được viết "vượt SOTA"** vào báo cáo. Notebook này chấm B1 và B2 trên đúng full test mà paper dùng.

**Điều kiện:** giữ y hệt B1/B2 hiện có. Cùng model 4-bit NF4, 512 token, prompt P0, greedy, `max_new_tokens` = 32, seed 42, L4. B2 dùng **đúng adapter** đã chấm ở commit 997f9c3; notebook kiểm sha256 trước khi chạy. Chỉ đổi file dữ liệu: `test.json` (10.028 câu, 3.353 ảnh). File này **chứa** 2.000 câu của tập con.

**Luật (chốt trước khi chạy):**
1. **So với ViTextBLIP-2:** chỉ được viết "cao hơn" nếu **cận dưới CI95 bootstrap** (10.000 lần, seed 42) của nhóm lớn hơn số công bố. Xét riêng EM và F1.
   - Giới hạn phải ghi kèm: paper không công bố cách chuẩn hoá khi tính EM/F1, nên kết luận chỉ đúng "với giả định cách chấm tương đương".
   - Paper có hai con số cho ViTextBLIP-2: Table 6 là 25,48 / 53,95; Table 10/12 là 24,83 / 52,51. So với số **cao hơn** (Table 6) để không có lợi cho nhóm.
2. **B2 vs B1 trên full test:** cải thiện nếu EM tăng **và** McNemar exact p < 0,05, như luật của B2.
3. **Kiểm tra tái lập (determinism):** 2.000 câu của tập con nằm trong full test. Trên đúng 2.000 câu đó, dự đoán của run full phải **trùng ≥ 99%** với run tập con đã có, vì cùng cấu hình và cùng GPU. Dưới 99% thì dừng và báo Claude **trước khi** dùng số liệu.
4. **Tập con có đại diện không:** báo cáo EM của tập con 2.000 câu so với CI95 của full test. Chỉ ghi nhận, không phải điều kiện dừng.

**Thời gian ước tính:**
- B1 ≈ 10.028 × 0,86 s ≈ **2,4 giờ**.
- B2 ≈ 10.028 × 1,01 s ≈ **2,8 giờ**.
- Tổng ≈**5–5,5 giờ, ≈9–10 CU**. Có thể chạy qua đêm (máy không Sleep, Chrome giữ tab Colab active).

**Nếu Colab rớt:** chạy lại từ phần 1 rồi **Run all**. Mỗi lô 500 câu được ghi lên Drive ngay khi xong, lô đã xong tự bỏ qua (mất tối đa ≈10 phút).

## 1. Thiết lập môi trường

Giống notebook 02/04 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất, `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

Kiểm trong vài giây:
- Config đúng bản chốt W07.
- `test.json` có đủ 10.028 câu và chứa trọn 2.000 câu của tập con.
- Adapter B2 trên Drive **đúng sha256** của adapter đã chấm (commit 997f9c3).
- Có sẵn kết quả B1/B2 trên tập con để kiểm tra tái lập.
- Selftest các script đạt.

*Kỳ vọng: `bfloat16 | 401408 | nf4 | bfloat16`, `full test: 10028 câu | 3353 ảnh | chứa 2000/2000 câu tập con`, `adapter sha256 khớp ✅`, hai dòng `SELFTEST OK`.*

In [ ]:
import json, yaml
from src.eval import paper_consistency as pc
from src.eval.run_baseline import adapter_fingerprint

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
CHUNK      = 500
RUN_B1 = True      # B1 full test (≈2,4 giờ)
RUN_B2 = True      # B2 full test (≈2,8 giờ)

FULL_TEST = f'{DRIVE_ROOT}/data/vitextvqa_official/test.json'
ADAPTER   = f'{DRIVE_ROOT}/adapters/W07_B2_qlora_r16_qwen25vl_vitextvqa_ep1_bf16_L4_seed42/epoch_1'
ADAPTER_SHA256 = '2e7b17d18b32b0f248b130dab4717b9b408b1da68e4fea5c3f5d6baaf9dcf863'   # adapter đã chấm ở commit 997f9c3
RUNS = {'B1': {'run': 'W07_B1_zeroshot_qwen25vl_vitextvqa_testfull_bf16_L4_seed42', 'adapter': None,
               'subset_run': 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42'},
        'B2': {'run': 'W07_B2_eval_qlora_r16_ep1_vitextvqa_testfull_bf16_L4_seed42', 'adapter': ADAPTER,
               'subset_run': 'W07_B2_eval_qlora_r16_ep1_vitextvqa_test2000_bf16_L4_seed42'}}
OUT_DIR = f'{EXP_ROOT}/W07_fulltest_summary_L4_seed42'

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
mdl, q = cfg['model'], cfg['quantization']
print('torch_dtype | max_pixels | quant | compute dtype')
print(f"{mdl['torch_dtype']} | {mdl['max_pixels']} | {q.get('bnb_4bit_quant_type')} | {q.get('bnb_4bit_compute_dtype')}")
assert (mdl['torch_dtype'], mdl['max_pixels'], q.get('bnb_4bit_quant_type'), q.get('bnb_4bit_compute_dtype')) \
       == ('bfloat16', 401408, 'nf4', 'bfloat16'), 'Config KHÔNG phải bản chốt W07 -> dừng, báo Claude'

full = json.load(open(FULL_TEST, encoding='utf-8'))
full_ids = {r['question_id'] for r in full}
SUB = {k: json.load(open(f"{EXP_ROOT}/{r['subset_run']}/predictions.json", encoding='utf-8')) for k, r in RUNS.items()}
sub_ids = {p['question_id'] for p in SUB['B1']}
print(f"full test: {len(full)} câu | {len({r['image'] for r in full})} ảnh | chứa {len(sub_ids & full_ids)}/{len(sub_ids)} câu tập con")
assert len(full) == 10028 and sub_ids <= full_ids, 'test.json không đúng -> dừng, báo Claude'

if RUN_B2:
    fp = adapter_fingerprint(ADAPTER)
    assert fp['weights_sha256'] == ADAPTER_SHA256, f"adapter trên Drive KHÁC adapter đã chấm ({fp['weights_sha256'][:12]}…) -> dừng, báo Claude"
    print(f"adapter sha256 khớp ✅ ({fp['weights_mb']} MB)")
for k, r in RUNS.items():
    print(f"  {k} -> {r['run']}")

!python -m src.eval.run_baseline --selftest 2>&1 | tail -n 1
!python -m src.eval.paper_consistency --selftest 2>&1 | tail -n 1

## 3. Chạy B1 rồi B2 trên full test (≈5–5,5 giờ)

Cùng script `run_baseline.py` như B1/B2. Chỉ thêm `--data_file` trỏ tới full test, còn B2 thêm `--adapter`. Mỗi run chạy bằng `!python` (tiến trình riêng). Run nào đã có `metrics.json` thì bỏ qua.

*Kỳ vọng: mỗi run 21 lô (`[lô k/21] xong 500 mẫu | … | EM lô = …`), cuối mỗi run `== KẾT QUẢ: EM=…`. B2 có thêm dòng `đã gắn adapter LoRA`. Bị ngắt: Run all lại, phải thấy `[lô …] đã có -> bỏ qua`.*

In [ ]:
for k, r in RUNS.items():
    if not {'B1': RUN_B1, 'B2': RUN_B2}[k]:
        print(f'{k}: tắt -> bỏ qua'); continue
    run_name = r['run']
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {k}: đã có metrics.json -> bỏ qua'); continue
    print(f'\n▶ {k}: {run_name}')
    if r['adapter']:
        !python -u -m src.eval.run_baseline --dataset vitextvqa_official --data_file {FULL_TEST} --run_name {run_name} --chunk {CHUNK} --adapter {ADAPTER}
    else:
        !python -u -m src.eval.run_baseline --dataset vitextvqa_official --data_file {FULL_TEST} --run_name {run_name} --chunk {CHUNK}

## 4. Kiểm tra tái lập trên 2.000 câu chung (luật 3) — XEM TRƯỚC khi đọc điểm

Lấy 2.000 câu của tập con từ run full, so từng câu với run tập con đã có.

*Kỳ vọng: `trùng ≥ 99%` cho cả B1 và B2, ΔEM ≈ 0. Nếu thấy ❌, dừng ở đây, chụp màn hình gửi Claude.*

In [ ]:
import pandas as pd
from src.eval.metrics import exact_match

def load(run):
    d = f'{EXP_ROOT}/{run}'
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')),
            json.load(open(f'{d}/predictions.json', encoding='utf-8')))

M, P, DET = {}, {}, {}
for k, r in RUNS.items():
    if not os.path.exists(f"{EXP_ROOT}/{r['run']}/metrics.json"):
        print(f'{k}: chưa có kết quả full test'); continue
    M[k], P[k] = load(r['run'])
    by_id = {p['question_id']: p for p in P[k]}
    same = [by_id[s['question_id']]['prediction'] == s['prediction'] for s in SUB[k]]
    em_sub = sum(exact_match(s['prediction'], s['answers']) for s in SUB[k]) / len(SUB[k])
    em_full_on_sub = sum(exact_match(by_id[s['question_id']]['prediction'], s['answers']) for s in SUB[k]) / len(SUB[k])
    DET[k] = {'n': len(same), 'identical_pct': 100 * sum(same) / len(same),
              'EM_subset_run': 100 * em_sub, 'EM_full_run_on_subset': 100 * em_full_on_sub}
    ok = DET[k]['identical_pct'] >= 99
    print(f"{k}: trùng {DET[k]['identical_pct']:.2f}% | EM tập con {DET[k]['EM_subset_run']:.2f} vs run full trên cùng câu "
          f"{DET[k]['EM_full_run_on_subset']:.2f} -> {'✅' if ok else '❌ DỪNG, báo Claude'}")
    DET[k]['pass'] = ok
if not all(d['pass'] for d in DET.values()):
    raise SystemExit('❌ Run full không tái lập run tập con (< 99% trùng) -> KHÔNG dùng số liệu; chụp màn hình gửi Claude.')

## 5. Kết quả full test + đối chiếu ViTextBLIP-2 + B2 vs B1

*Kỳ vọng: Bảng 1 (điểm), Bảng 2 (so với ViTextBLIP-2 theo luật 1, có CI95), dòng `➜ KẾT LUẬN B2 vs B1 (full test)`, và dòng về độ đại diện của tập con.*

In [ ]:
rows = []
for k in P:
    mm = M[k]['metrics']
    rows.append({'run': {'B1': 'B1 zero-shot', 'B2': 'B2 QLoRA 1 epoch'}[k], 'n': M[k]['meta']['num_samples'],
                 'EM': 100 * mm['exact_match'], 'F1 (phụ)': 100 * mm['token_f1'], 'ANLS': 100 * mm['anls'],
                 'VQA-Acc': 100 * mm['vqa_accuracy'], 's/câu': M[k]['meta']['sec_per_sample'],
                 'giờ suy luận': round(M[k]['meta']['inference_time_sec'] / 3600, 2)})
rows.append({'run': 'ViTextBLIP-2 (paper, Table 6)', 'n': 'full test', 'EM': pc.TABLE6_VITEXTBLIP2['em'],
             'F1 (phụ)': pc.TABLE6_VITEXTBLIP2['f1']})
print('==== Bảng 1. ViTextVQA FULL test (10.028 câu) ====')
display(pd.DataFrame(rows).round(2))

SOTA = {k: pc.vitextblip2_report(P[k]) for k in P}
print('\n==== Bảng 2. So với ViTextBLIP-2 (luật 1: cận dưới CI95 > số công bố) ====')
display(pd.DataFrame([{'run': k, 'metric': met, 'nhóm': SOTA[k][met]['ours_pct'], 'CI95': SOTA[k][met]['ours_ci95_pct'],
                       'paper': SOTA[k][met]['published_pct'], 'chênh (điểm)': SOTA[k][met]['diff_points'],
                       'kết luận': SOTA[k][met]['verdict']} for k in SOTA for met in ('EM', 'F1')]))
print('  lưu ý:', pc.TABLE6_VITEXTBLIP2['note'], '| cách chuẩn hoá của paper không công bố')

CMP = None
if 'B1' in P and 'B2' in P:
    out = f'{OUT_DIR}/paired_B1_vs_B2_vitextvqa_full.json'
    ra, rb = f"{EXP_ROOT}/{RUNS['B1']['run']}", f"{EXP_ROOT}/{RUNS['B2']['run']}"
    !python src/eval/compare_runs.py --run_a {ra} --run_b {rb} --label_a B1 --label_b B2 --out {out}
    CMP = json.load(open(out, encoding='utf-8'))
    verdict = ('CẢI THIỆN (EM tăng, p < 0,05)' if CMP['em_diff'] > 0 and CMP['p_value'] < 0.05 else
               'KÉM HƠN (EM giảm, p < 0,05)' if CMP['em_diff'] < 0 and CMP['p_value'] < 0.05 else
               'CHƯA KẾT LUẬN ĐƯỢC (p ≥ 0,05)')
    print(f"➜ KẾT LUẬN B2 vs B1 (full test): {verdict} | chỉ B1 đúng {CMP['b']} | chỉ B2 đúng {CMP['c']} | p = {CMP['p_value']:.2g}")

REP = {}
for k in P:
    lo, hi = SOTA[k]['EM']['ours_ci95_pct']
    em_sub = DET[k]['EM_subset_run']
    REP[k] = {'EM_subset2000': em_sub, 'EM_full': SOTA[k]['EM']['ours_pct'], 'EM_full_ci95': [lo, hi],
              'subset_inside_full_ci95': lo <= em_sub <= hi}
    print(f"{k}: EM tập con 2.000 = {em_sub:.2f} | full = {SOTA[k]['EM']['ours_pct']:.2f} CI95 [{lo:.2f}; {hi:.2f}] -> "
          f"{'tập con đại diện tốt' if REP[k]['subset_inside_full_ci95'] else 'tập con lệch khỏi CI95 của full (ghi chú trong báo cáo)'}")

## 6. Lưu tóm tắt

Ghi `experiments/W07_fulltest_summary_L4_seed42/fulltest_summary.json`. Cursor chép:
- File tóm tắt này và `paired_B1_vs_B2_vitextvqa_full.json`.
- `metrics.json`, `predictions.json`, `run_signature.json` của hai run full test.

**Không** chép `chunks/`.

*Kỳ vọng: `Ghi -> …/fulltest_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
summary = {
    'what': 'B1 (zero-shot) và B2 (QLoRA 1 epoch) trên toàn bộ ViTextVQA test 10.028 câu; cùng cấu hình B1/B2 trên tập con',
    'pre_registered_rules': {
        'vs_vitextblip2': 'CAO HƠN nếu cận dưới CI95 bootstrap (10.000, seed 42) > số công bố Table 6 (EM 25,48 / F1 53,95)',
        'B2_vs_B1': 'cải thiện nếu EM tăng và McNemar exact p < 0,05',
        'determinism': '>= 99% dự đoán trùng run tập con trên 2.000 câu chung',
        'representativeness': 'ghi nhận EM tập con có nằm trong CI95 EM full không'},
    'full_test_file': 'data/vitextvqa_official/test.json', 'adapter_sha256': ADAPTER_SHA256,
    'runs': {k: r['run'] for k, r in RUNS.items() if k in P},
    'table1': rows, 'vs_vitextblip2': SOTA, 'reference': pc.TABLE6_VITEXTBLIP2, 'source': pc.PAPER_SOURCE,
    'paired_B1_vs_B2': {kk: vv for kk, vv in CMP.items() if kk != 'only_b_question_ids'} if CMP else None,
    'determinism_vs_subset_runs': DET, 'subset_representativeness': REP, 'gpu': GPU,
}
with open(f'{OUT_DIR}/fulltest_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/fulltest_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')